## CHMM Distillation - Quadratic Log-Rank Clone Schedule


## Clone schedule used here

- Tokens are ranked by observed frequency in the sampled LVD corpus.
- The schedule uses a quadratic function of log-rank:

  $$\log(\mathrm{freq}) \approx a \log(r)^2 + b \log(r) + c$$

  where \(r\) is the token rank after sorting tokens by frequency. This is inspired by Zipf's law, which would be a straight line in log-log space. but here an additional \((\log r)^2\) term is included so the curve can bend and better match the observed token distribution.
- Clone counts are then scaled from **20 clones** for the highest-frequency tokens down to **2 clones** for the lowest-ranked token, and rounded up with `ceil`.

In [1]:
import os
from pathlib import Path
from transformers import AutoTokenizer

BASE_MODEL_PATH = 'gpt2-large'
TOKENIZER_NAME_OR_PATH = BASE_MODEL_PATH
INPUT_FILE = './workspace/inference_data/distillation_prompts.json'

DATASET = 'gpt2-large'
DATA_PATH = f'./workspace/hmm_data/{DATASET}'
CUDA_CORES = '0'

LVD_CHUNK_SIZE = 100000
TRAIN_CHUNK_SIZE = 100000
TRAIN_TOTAL_CHUNKS = 100
DEV_CHUNK_SIZE = 10000
MAX_NEW_TOKENS = 32
BATCH_SIZE = 8192

Path(DATA_PATH).mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME_OR_PATH)
VOCAB_SIZE = tokenizer.vocab_size
EOS_TOKEN_ID = tokenizer.eos_token_id

print({'data_path': DATA_PATH, 'vocab_size': VOCAB_SIZE, 'eos_token_id': EOS_TOKEN_ID})


{'data_path': './workspace/hmm_data/gpt2-large', 'vocab_size': 50257, 'eos_token_id': 50256}


In [2]:
import numpy as np
import torch

LVD_FILE = Path(DATA_PATH) / f'{DATASET}.lvd'
FIT_SAMPLE_LENGTH = None


def load_lvd_sequences(path: Path, sample_length: int | None = None, mmap: bool = True) -> torch.Tensor:
    load_kwargs = {'map_location': 'cpu', 'weights_only': True}
    if mmap:
        load_kwargs['mmap'] = True

    try:
        sequences = torch.load(path, **load_kwargs).long()
    except TypeError:
        load_kwargs.pop('mmap', None)
        sequences = torch.load(path, **load_kwargs).long()

    if sample_length is not None:
        sequences = sequences[:, :sample_length]
    return sequences.contiguous()


if not LVD_FILE.exists():
    raise FileNotFoundError(
        f'Missing required LVD file: {LVD_FILE}. '
        'Generate the sampled LVD corpus first, or update DATA_PATH.'
    )

lvd_sequences = load_lvd_sequences(LVD_FILE, sample_length=FIT_SAMPLE_LENGTH, mmap=True)
flattened = lvd_sequences.reshape(-1)
flattened = flattened[flattened >= 0]
token_frequency = torch.bincount(flattened, minlength=VOCAB_SIZE)[:VOCAB_SIZE]

sorted_counts, _ = torch.sort(token_frequency, descending=True)
ranks = np.arange(1, len(sorted_counts) + 1, dtype=float)
counts = sorted_counts.numpy().astype(float)

positive = counts > 0
fit_x = ranks[positive]
fit_y = counts[positive]
log_y = np.log(fit_y)

quad_coef = np.polyfit(np.log(fit_x), log_y, 2)
FITTED_CURVE_A = float(quad_coef[0])
FITTED_CURVE_B = float(quad_coef[1])
FITTED_CURVE_C = float(quad_coef[2])

quad_log_pred = np.polyval(quad_coef, np.log(fit_x))
residual = log_y - quad_log_pred
sse = float(np.sum(residual ** 2))
sst = float(np.sum((log_y - log_y.mean()) ** 2))

print('Fitted quadratic log-rank curve:')
print(f'log(freq) ~= a * log(rank)^2 + b * log(rank) + c')
print(f'a = {FITTED_CURVE_A:.4f}')
print(f'b = {FITTED_CURVE_B:.4f}')
print(f'c = {FITTED_CURVE_C:.3f}')

Fitted quadratic log-rank curve:
log(freq) ~= a * log(rank)^2 + b * log(rank) + c
a = -0.1134
b = 0.4477
c = 8.387


In [3]:
import torch

def build_clone_schedule(
    token_frequency: torch.Tensor,
    max_count: int = 20,
    min_count: int = 2,
    curve_a: float = -0.1134,
    curve_b: float = 0.4477,
    curve_c: float = 8.387,
    **_: object,
) -> torch.Tensor:
    vocab_size = token_frequency.shape[0]
    ranking = sorted(range(vocab_size), key=lambda idx: (-int(token_frequency[idx]), idx))

    ranks = torch.arange(1, vocab_size + 1, dtype=torch.float64)
    log_ranks = torch.log(ranks)
    curve = curve_a * log_ranks.square() + curve_b * log_ranks + curve_c

    curve = torch.flip(torch.cummax(torch.flip(curve, dims=[0]), dim=0).values, dims=[0])
    denom = torch.clamp(curve[0] - curve[-1], min=1e-12)
    normalized = (curve - curve[-1]) / denom
    clone_values = min_count + (max_count - min_count) * normalized
    ranked_clones = torch.ceil(clone_values).to(torch.long).clamp(min_count, max_count)

    clones = torch.empty((vocab_size,), dtype=torch.long)
    clones[torch.tensor(ranking, dtype=torch.long)] = ranked_clones
    return clones


CLONE_SCHEDULE_FUNCTION = './tutorial_distillation_chmm_quadratic_log.ipynb:build_clone_schedule'


In [4]:
import shlex
import subprocess

MODEL_ID = f'chmm_{DATASET}_quadratic_log_20to2'
MODEL_PATH = f'./workspace/models/{MODEL_ID}'
LOG_FILE = f'./workspace/logs/{MODEL_ID}.log'
TRAINING_LOG_FILE = f'./workspace/logs/{MODEL_ID}_training.log'
PID_FILE = f'./workspace/logs/{MODEL_ID}.pid'
EM_SCHEDULE = '10,1;5,2;4,5;4,10;4,20;1,40'

os.makedirs('./workspace/logs', exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)

cmd = f'''torchrun --standalone --nproc_per_node={len(CUDA_CORES.split(','))} train_chmm.py \
    --model_path {MODEL_PATH} \
    --checkpoint 0 --save_per_step 10 \
    --data_path {DATA_PATH} --dataset {DATASET} --total_chunks {TRAIN_TOTAL_CHUNKS} \
    --batch_size {BATCH_SIZE} --sample_length {MAX_NEW_TOKENS} \
    --em_schedule "{EM_SCHEDULE}" \
    --vocab_size {VOCAB_SIZE} --eos_token_id {EOS_TOKEN_ID} \
    --clone_schedule_file {DATA_PATH}/{DATASET}.lvd \
    --clone_schedule_function {CLONE_SCHEDULE_FUNCTION} \
    --pair_code_chunk_count 0 \
    --dropout 0.0 --pseudocount 0.0003 \
    --log_file {LOG_FILE}'''.strip()

env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = CUDA_CORES
env['PYTHONUNBUFFERED'] = '1'

print('Launching:')
print(cmd)
print(f'\nProgress log file: {LOG_FILE}')
print(f'Training log file: {TRAINING_LOG_FILE}')
print(f'PID file: {PID_FILE}')

log_f = open(TRAINING_LOG_FILE, 'a', buffering=1)

proc = subprocess.Popen(
    shlex.split(cmd),
    stdout=log_f,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
    env=env,
    start_new_session=True,
    cwd=os.getcwd(),
)

Path(PID_FILE).write_text(str(proc.pid))
log_f.close()
print(f'Started background job with PID {proc.pid}')


Launching:
torchrun --standalone --nproc_per_node=1 train_chmm.py     --model_path ./workspace/models/chmm_gpt2-large_quadratic_log_20to2     --checkpoint 0 --save_per_step 10     --data_path ./workspace/hmm_data/gpt2-large --dataset gpt2-large --total_chunks 100     --batch_size 8192 --sample_length 32     --em_schedule "10,1;5,2;4,5;4,10;4,20;1,40"     --vocab_size 50257 --eos_token_id 50256     --clone_schedule_file ./workspace/hmm_data/gpt2-large/gpt2-large.lvd     --clone_schedule_function ./tutorial_distillation_chmm_quadratic_log.ipynb:build_clone_schedule     --pair_code_chunk_count 0     --dropout 0.0 --pseudocount 0.0003     --log_file ./workspace/logs/chmm_gpt2-large_quadratic_log_20to2.log

Progress log file: ./workspace/logs/chmm_gpt2-large_quadratic_log_20to2.log
Training log file: ./workspace/logs/chmm_gpt2-large_quadratic_log_20to2_training.log
PID file: ./workspace/logs/chmm_gpt2-large_quadratic_log_20to2.pid
Started background job with PID 320898
